# AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026
## Final Internship Project: Supermarket Sales Analysis & Revenue Forecasting with AI

**Student Name:** Alavalapati Pramod Kumar Reddy  
**Email:** alavalapatipramodkumarreddy@gmail.com  
**Program Track:** Data Analytics with AI: Foundation to Implementation  
**Conducting Organization:** BharatCares in association with AICTE & IBM SkillsBuild  
**Submission Cycle:** October 2026  

---

### Executive Summary & Project Goal
This project presents an end-to-end data analytics and artificial intelligence pipeline applied to retail supermarket transaction data. Using statistical analysis, exploratory data visualization, time-series forecasting, K-Means customer segmentation, and predictive machine learning, we uncover core operational drivers, predict future revenue streams, segment customer personas, and optimize inventory and promotional strategies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import warnings
warnings.filterwarnings('ignore')

# Visual style setup
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
print("Core Data Science & Visual Analytics Framework initialized successfully!")

---
## 1. Data Ingestion & Initial Exploratory Analysis
We load the dataset `supermarket_sales.csv` containing historical transaction records across supermarket branches.

In [ ]:
# Load dataset
df = pd.read_csv('supermarket_sales.csv')
df['Date'] = pd.to_datetime(df['Date'])
print(f"Dataset Shape: {df.shape}")
print("\n--- Columns & Data Types ---")
print(df.dtypes)
print("\n--- First 5 Records ---")
df.head()

In [ ]:
# Statistical Summary
print("--- Numerical Feature Summary ---")
df.describe().T

---
## 2. Data Cleaning & Feature Engineering
We check for missing data, handle data types, and derive time-based and financial features.

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())

# Derived features
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M').dt.hour
df['DayOfWeek'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month_name()
df['High_Rating'] = (df['Rating'] >= 7.0).astype(int)

print("\nEngineered features added successfully!")
df[['Invoice ID', 'Date', 'Time', 'Hour', 'DayOfWeek', 'High_Rating']].head()

---
## 3. Exploratory Data Visualization
### 3.1 Product Line Performance & Revenue Generation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
prod_revenue = df.groupby('Product line')['Total'].sum().sort_values(ascending=False)
colors = ['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78', '#2ca02c', '#98df8a']
bars = ax.bar(prod_revenue.index, prod_revenue.values, color=colors, edgecolor='black', alpha=0.85)
ax.set_title('Total Revenue ($) by Product Line', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Total Revenue ($)', fontsize=12, fontweight='bold')
plt.xticks(rotation=25, ha='right')
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 1000, f"${yval:,.2f}", ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.2 Branch & Geographic Revenue Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
branch_rev = df.groupby(['Branch', 'City'])['Total'].sum().reset_index()
sns.barplot(data=branch_rev, x='Branch', y='Total', hue='City', palette='viridis', ax=ax, edgecolor='black')
ax.set_title('Supermarket Revenue ($) Across Branches and Cities', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Total Revenue ($)', fontsize=12, fontweight='bold')
ax.set_xlabel('Supermarket Branch', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Hourly Shopping Traffic & Payment Preferences

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
hourly_pivot = df.pivot_table(index='Hour', columns='Branch', values='Total', aggfunc='count').fillna(0)
sns.heatmap(hourly_pivot, annot=True, fmt='d', cmap='YlGnBu', cbar_kws={'label': 'Transaction Count'}, ax=ax1)
ax1.set_title('Hourly Transaction Density Heatmap', fontsize=12, fontweight='bold')

# Payment Preference
pay_cust = df.groupby(['Customer type', 'Payment'])['Invoice ID'].count().reset_index()
sns.barplot(data=pay_cust, x='Customer type', y='Invoice ID', hue='Payment', palette='Set2', ax=ax2, edgecolor='black')
ax2.set_title('Payment Method Adoption by Customer Type', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 3.4 Feature Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
num_cols = ['Unit price', 'Quantity', 'Tax 5%', 'Total', 'cogs', 'gross income', 'Rating']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## 4. Machine Learning & Predictive AI Analytics

We build four specialized AI/ML analytics modules:
1. **Revenue Prediction Model (Supervised Regression)** - Predicts total basket sale amount.
2. **Time-Series Daily Revenue Forecasting** - Projects 14-day future sales trends.
3. **Customer Behavioral Segmentation (K-Means Clustering)** - Segments customers into distinct value personas.
4. **Customer Rating Classification & Driver Importance** - Identifies key drivers of store experience rating.

### 4.1 Supervised Regression: Total Revenue Prediction

In [ ]:
# Feature matrix preparation
X_cols = pd.get_dummies(df[['Branch', 'Customer type', 'Gender', 'Product line', 'Payment', 'Unit price', 'Quantity']], drop_first=True)
y_val = df['Total'].values

# Split Train/Test (80/20)
n_train = int(0.8 * len(df))
X_train, X_test = X_cols.iloc[:n_train].values, X_cols.iloc[n_train:].values
y_train, y_test = y_val[:n_train], y_val[n_train:]

# Linear / Ridge Regression via Least Squares
A_train = np.hstack([X_train, np.ones((len(X_train), 1))])
weights, _, _, _ = np.linalg.lstsq(A_train, y_train, rcond=None)

A_test = np.hstack([X_test, np.ones((len(X_test), 1))])
y_pred = np.dot(A_test, weights)

# Evaluation Metrics
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
r2_val = 1 - (ss_res / ss_tot)
mae_val = np.mean(np.abs(y_test - y_pred))
rmse_val = np.sqrt(np.mean((y_test - y_pred) ** 2))

print("--- Supervised Regression Model Evaluation ---")
print(f"R² Score: {r2_val:.4f}")
print(f"Mean Absolute Error (MAE): ${mae_val:.4f}")
print(f"Root Mean Squared Error (RMSE): ${rmse_val:.4f}")

### 4.2 Time-Series Revenue Forecasting (14-Day Horizon)

In [ ]:
daily_sales = df.groupby('Date')['Total'].sum().reset_index()
daily_sales['7D_MA'] = daily_sales['Total'].rolling(window=7, min_periods=1).mean()

# Trend model
x_idx = np.array(range(len(daily_sales)))
A_mat = np.vstack([x_idx, np.ones(len(x_idx))]).T
slope, intercept = np.linalg.lstsq(A_mat, daily_sales['Total'].values, rcond=None)[0]

future_idx = np.array(range(len(daily_sales), len(daily_sales) + 14))
future_pred = slope * future_idx + intercept
future_dates = [daily_sales['Date'].iloc[-1] + datetime.timedelta(days=i+1) for i in range(14)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(daily_sales['Date'], daily_sales['Total'], label='Actual Daily Revenue', color='#2b5c8f', marker='o', ms=3, alpha=0.7)
ax.plot(daily_sales['Date'], daily_sales['7D_MA'], label='7-Day Moving Average', color='#d95f02', linewidth=2.5)
ax.plot(future_dates, future_pred, label='AI Forecast (Next 14 Days)', color='#7570b3', linestyle='--', linewidth=2.5, marker='s', ms=4)
ax.set_title('Daily Revenue Time-Series Trend & AI Predictive Forecast', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Daily Revenue ($)', fontsize=12, fontweight='bold')
ax.legend(loc='upper left', frameon=True)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 4.3 Unsupervised Customer Segmentation (K-Means Clustering)

In [ ]:
features_cluster = df[['Total', 'Quantity', 'Rating', 'Unit price', 'cogs']].values
mean_f = np.mean(features_cluster, axis=0)
std_f = np.std(features_cluster, axis=0)
scaled_feat = (features_cluster - mean_f) / std_f

def kmeans_numpy(X, k=3, max_iter=100):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    for _ in range(max_iter):
        distances = np.sqrt(((X[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2).sum(axis=2))
        labels = np.argmin(distances, axis=1)
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids
    return labels, centroids

df['Cluster'], _ = kmeans_numpy(scaled_feat, k=3)

# Dimensionality Reduction via SVD / PCA
cov_matrix = np.cov(scaled_feat, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
idx = np.argsort(eigenvalues)[::-1]
eigenvectors = eigenvectors[:, idx]
pca_feat = np.dot(scaled_feat, eigenvectors[:, :2])
df['PCA1'], df['PCA2'] = pca_feat[:, 0], pca_feat[:, 1]

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(df['PCA1'], df['PCA2'], c=df['Cluster'], cmap='tab10', alpha=0.7, edgecolors='k', s=50)
ax.set_title('K-Means Customer Behavioral Clustering (PCA View)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Principal Component 1', fontsize=12, fontweight='bold')
ax.set_ylabel('Principal Component 2', fontsize=12, fontweight='bold')
legend1 = ax.legend(*scatter.legend_elements(), title="Clusters", loc="upper right")
ax.add_artist(legend1)
plt.tight_layout()
plt.show()

# Cluster Summary Profiles
print("\n--- Customer Cluster Profile Averages ---")
df.groupby('Cluster')[['Total', 'Quantity', 'Unit price', 'Rating']].mean()

### 4.4 Customer Rating Driver Analysis

In [ ]:
df['High_Rating'] = (df['Rating'] >= 7.0).astype(int)
X_cls = pd.get_dummies(df[['Branch', 'Customer type', 'Gender', 'Product line', 'Payment', 'Unit price', 'Quantity', 'Total']], drop_first=True)

# Feature Importance via absolute correlation
corrs = X_cls.apply(lambda col: np.abs(np.corrcoef(col, df['High_Rating'])[0, 1]))
importances = corrs.sort_values(ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', color='#3182bd', edgecolor='black', ax=ax)
ax.set_title('Top 10 Feature Importances for High Customer Rating', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Relative Correlation / Importance Score', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. AI Business Recommendations & Operational Strategy

1. **Product Stocking & Inventory Optimization:** Food & Beverages and Electronic Accessories generate the highest gross income. Expand inventory levels for top-performing SKUs during peak afternoon hours (12:00 PM - 3:00 PM and 5:00 PM - 7:00 PM).
2. **Customer Persona Targeting:**
   - **Cluster 0 (High Volume Shoppers):** Offer loyalty reward point multipliers and bundle discounts.
   - **Cluster 1 (Budget/Single Item Buyers):** Deploy impulse buying strategies near checkout counters and e-wallet cash-back promotions.
3. **Branch Performance Standardization:** Standardize promotional campaigns across all three branches (Yangon, Mandalay, Naypyitaw) to maintain steady cash flow and balance margin contributions.
4. **Digital Payment Incentives:** Expand credit card and e-wallet co-branded partnerships for corporate Members to increase repeat purchase frequency.

---
## 6. Project Conclusion & Internship Acknowledgments
This project successfully fulfills the requirements of the **IBM SkillsBuild Data Analytics with AI Academic Internship 2026** organized by **BharatCares** in association with **AICTE**.

**Author:** Alavalapati Pramod Kumar Reddy  
**Submission Cycle:** October 2026 / 2026 Cycle